In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
constraints = """torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57
"""
with open("/content/constraints.txt", "w") as f:
    f.write(constraints)

In [3]:
!wget -qO- https://astral.sh/uv/install.sh | sh

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!uv pip install --system torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

!uv pip install --system \
    sympy numpy transformers vllm tqdm \
    antlr4-python3-runtime==4.11.1 accelerate \
    -c /content/constraints.txt

downloading uv 0.11.12 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.12.13 environment at: /usr
Checked 3 packages in 104ms
Using Python 3.12.13 environment at: /usr
Checked 7 packages in 109ms


In [4]:
# If vllm import fails the first time after install, restart the session and re-run all cells. Same for any persistent setup errors.
import torch
import vllm
print("Post-restart checks:")
print("  torch:", torch.__version__)
print("  CUDA available:", torch.cuda.is_available())
print("  Device:", torch.cuda.get_device_name(0))
print("  vllm:", vllm.__version__)
print("  GPU memory free:", round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), "GB")

Post-restart checks:
  torch: 2.5.1+cu121
  CUDA available: True
  Device: NVIDIA A100-SXM4-80GB
  vllm: 0.7.3
  GPU memory free: 84.65 GB


In [5]:
from pathlib import Path
import sys
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/cse151b")
GIVEN_DATA_DIR = PROJECT_DIR / "given_data"
OUTPUT_DIR = PROJECT_DIR  # final outputs land here on Drive

# Local hot-path directory (Colab disk — fast, reliable, but wiped on runtime death)
LOCAL_DIR = Path("/content/local_results")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Inputs
DATA_PATH = str(GIVEN_DATA_DIR / "public.jsonl")
# DATA_PATH = str(GIVEN_DATA_DIR / "private.jsonl")

# Hot-path outputs — written to local disk during generation
RESPONSES_PATH = LOCAL_DIR / "responses.jsonl"
LOG_PATH = LOCAL_DIR / "responses.log"

# Drive backup of responses (snapshot target during run, restore source after restart)
RESPONSES_BACKUP = OUTPUT_DIR / "responses.jsonl"

# Final outputs — written to Drive (only one write each, at the end of their step)
SCORED_PATH = OUTPUT_DIR / "scored_results.jsonl"
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

# Make given_data importable
sys.path.insert(0, str(GIVEN_DATA_DIR))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert OUTPUT_DIR.exists(), f"Drive not mounted? {OUTPUT_DIR} missing"

# If a Drive backup exists from a previous session but no local copy, restore it
if RESPONSES_BACKUP.exists() and not RESPONSES_PATH.exists():
    shutil.copy2(RESPONSES_BACKUP, RESPONSES_PATH)
    print(f"Restored {RESPONSES_PATH.stat().st_size} bytes from Drive backup")

In [24]:
import csv
import json
import re
import time
import random
import pandas as pd
import numpy as np
import gc
from scipy.stats import bootstrap
from collections import Counter
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"

# Set GPU_ID earlier if training off of colab and have multiple GPUs. Need to declare before torch is imported.
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

In [7]:
with open(DATA_PATH) as f:
    data = [json.loads(line) for line in f]

In [8]:
n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


In [9]:
# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))


── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


In [10]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [11]:
# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



In [12]:
SEED = 42
# Parameters to be tested
MAX_TOKENS = 16384
MAX_MODEL_LEN = 16384
MAX_NUM_BATCHED_TOKENS = 16384
MAX_NUM_SEQS = 32

VOTERS = 1

TEMPERATURE = 0.6

# Testing on small batches
EXPERIMENT_DIR = Path("/content/drive/MyDrive/cse151b/experiments")
NUM_SAMPLES = 50

random.seed(SEED)

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [14]:

# llm = LLM(
#     model=MODEL_ID,
#     dtype="bfloat16",
#     trust_remote_code=True,
#     max_model_len=MAX_MODEL_LEN,
#     gpu_memory_utilization=0.92,
#     max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
#     max_num_seqs=MAX_NUM_SEQS,
#     enable_prefix_caching=True,
#     enable_chunked_prefill=True,
#     disable_log_stats=False,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     n=VOTERS,
#     temperature=TEMPERATURE,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
#     seed=SEED,
#     stop=["<|im_end|>", "<|endoftext|>"],
# )

# print("Model loaded.")

In [15]:
import utils
from utils import last_boxed_only_string, remove_boxed
from judger import Judger

In [16]:
def extract_letter(text: str) -> str:
    # Strip thinking trace — only look at content after </think>
    think_end = text.rfind("</think>")
    search_text = text[think_end + len("</think>"):] if think_end >= 0 else text

    # First try: pull the last \boxed{...} content using utils' brace-aware parser
    boxed = last_boxed_only_string(search_text)
    if boxed is not None:
        inner = remove_boxed(boxed)
        if inner:
            m = re.search(r"[A-Za-z]", inner)
            if m:
                return m.group(0).upper()
    # Fallback: last standalone capital letter in the post-think response
    matches = re.findall(r"\b([A-Z])\b", search_text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

In [17]:
judger = Judger(strict_extract=False)

In [18]:
#
#
# NEW EXPERIMENT TESTING BEGINS HERE
#
# Besides changing model params
#
#

In [19]:
# Pick first NUM_SAMPLES items for fast experiments — adjust to taste
EVAL_SUBSET = random.sample(data, NUM_SAMPLES)
print(f"Experiment subset: {len(EVAL_SUBSET)} items")

def extract_freeform_answer(text):
    """Pull the last boxed answer out of a response (post-think)."""
    think_end = text.rfind("</think>")
    search = text[think_end + len("</think>"):] if think_end >= 0 else text
    boxed = last_boxed_only_string(search)
    if boxed:
        return remove_boxed(boxed) or ""
    return ""

def evaluate(items, responses_per_item):
    """Score a list of (item, response) pairs. responses_per_item can be a single response or a list."""
    n_correct = 0
    details = []
    for item, resp in zip(items, responses_per_item):
        if isinstance(resp, list):
            # Self-consistency: vote across responses
            answers = [extract_freeform_answer(r) if not item.get("options")
                       else extract_letter(r) for r in resp]
            # Majority vote, ignoring empty answers
            non_empty = [a for a in answers if a]
            if not non_empty:
                pred, correct = "", False
            else:
                pred = Counter(non_empty).most_common(1)[0][0]
                if item.get("options"):
                    correct = pred == str(item["answer"]).strip().upper()
                else:
                    gold = item["answer"]
                    gold_list = gold if isinstance(gold, list) else [gold]
                    try:
                        correct = judger.auto_judge(pred=f"\\boxed{{{pred}}}", gold=gold_list,
                                                    options=[[]] * len(gold_list))
                    except Exception:
                        correct = False
            details.append({"id": item.get("id"), "pred": pred, "correct": correct, "n_samples": len(resp)})
        else:
            # Single response
            if item.get("options"):
                correct = score_mcq(resp, str(item["answer"]))
            else:
                gold = item["answer"]
                gold_list = gold if isinstance(gold, list) else [gold]
                try:
                    correct = judger.auto_judge(pred=resp, gold=gold_list,
                                                options=[[]] * len(gold_list))
                except Exception:
                    correct = False
            details.append({"id": item.get("id"), "correct": correct})
        n_correct += int(correct)
    return n_correct / len(items), details

def make_prompt_with_system(item, system, user_prefix=""):
    """Build a prompt with custom system + optional user prefix."""
    _, user = build_prompt(item["question"], item.get("options"))
    full_user = user_prefix + user if user_prefix else user
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": full_user}],
        tokenize=False, add_generation_prompt=True,
    )

Experiment subset: 50 items


In [20]:
# # Baseline: current production settings on the subset
# # USE IF TESTING PARAMS
# # sampling_base = SamplingParams(max_tokens=12288, temperature=0.6, top_p=0.95, top_k=20)

# prompts = [make_prompt_with_system(item, build_prompt(item["question"], item.get("options"))[0])
#            for item in EVAL_SUBSET]
# outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=True)
# baseline_responses = [out.outputs[0].text.strip() for out in outputs]

In [21]:
# acc_base, _ = evaluate(EVAL_SUBSET, baseline_responses)
# print(f"Baseline on {len(EVAL_SUBSET)} items: {acc_base*100:.1f}%")

In [27]:
"""
vLLM Hyperparameter Sweeps for Qwen3-4B-Thinking on Math Inference
====================================================================

  GROUP A (Cell 1): Sampling-param sweeps — share ONE LLM instance.
  GROUP B (Cell 2): Serving-param sweeps — RELOAD LLM per config.
                    Each serving param tested in isolation:
                      - max_num_seqs
                      - max_num_batched_tokens
                      - max_model_len
                    Baseline: (max_num_seqs=32, max_num_batched_tokens=16384,
                               max_model_len=16384, max_num_tokens=16384)

Each group saves its own Excel file independently.
"""

# ============================================================
# Shared utilities
# ============================================================

def build_prompts(eval_subset):
    return [
        make_prompt_with_system(item, build_prompt(item["question"], item.get("options"))[0])
        for item in eval_subset
    ]


def run_generation(llm, prompts, sampling_params, eval_subset):
    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=True)
    elapsed = time.time() - t0

    responses = [out.outputs[0].text.strip() for out in outputs]
    acc, correct_flags = evaluate(eval_subset, responses)

    all_samples = [s for out in outputs for s in out.outputs]
    truncated = sum(1 for s in all_samples if s.finish_reason == "length")
    total = len(all_samples)
    lengths = [len(s.token_ids) for s in all_samples]

    binary_flags = [int(f["correct"]) for f in correct_flags]
    ci = bootstrap(
        (np.array(binary_flags, dtype=float),),
        np.mean,
        confidence_level=0.95,
        n_resamples=10000,
        method="percentile",
    )

    return {
        "accuracy": acc,
        "ci_low": float(ci.confidence_interval.low),
        "ci_high": float(ci.confidence_interval.high),
        "trunc_rate": truncated / max(total, 1),
        "avg_gen_len": float(np.mean(lengths)),
        "p95_gen_len": float(np.percentile(lengths, 95)),
        "wall_time_s": elapsed,
    }


def print_row(name, value, m):
    print(
        f"{name}={value}: "
        f"acc={m['accuracy']:.3f} [{m['ci_low']:.3f}, {m['ci_high']:.3f}], "
        f"trunc={m['trunc_rate']:.1%}, "
        f"p95_len={m['p95_gen_len']:.0f}, "
        f"time={m['wall_time_s']:.1f}s"
    )


def cleanup_llm(llm):
    del llm
    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# GROUP A: SAMPLING-PARAM SWEEPS (one shared LLM)
# ============================================================

def sweep_max_tokens(llm, prompts, eval_subset):
    results = []
    for mt in [4096, 8192, 16384]:
        sp = SamplingParams(
            max_tokens=mt, n=1,
            temperature=TEMPERATURE, top_p=0.95, top_k=20, min_p=0.0,
            seed=SEED, stop=["<|im_end|>", "<|endoftext|>"],
        )
        m = run_generation(llm, prompts, sp, eval_subset)
        m["max_tokens"] = mt
        results.append(m)
        print_row("max_tokens", mt, m)
    return pd.DataFrame(results)


def sweep_temperature(llm, prompts, eval_subset):
    results = []
    for T in [0.3, 0.6, 0.9]:
        sp = SamplingParams(
            max_tokens=MAX_TOKENS, n=1,
            temperature=T, top_p=0.95, top_k=20, min_p=0.0,
            seed=SEED, stop=["<|im_end|>", "<|endoftext|>"],
        )
        m = run_generation(llm, prompts, sp, eval_subset)
        m["temperature"] = T
        results.append(m)
        print_row("temperature", T, m)
    return pd.DataFrame(results)


def sweep_top_p(llm, prompts, eval_subset):
    results = []
    for tp in [0.9, 0.95, 1.0]:
        sp = SamplingParams(
            max_tokens=MAX_TOKENS, n=1,
            temperature=TEMPERATURE, top_p=tp, top_k=20, min_p=0.0,
            seed=SEED, stop=["<|im_end|>", "<|endoftext|>"],
        )
        m = run_generation(llm, prompts, sp, eval_subset)
        m["top_p"] = tp
        results.append(m)
        print_row("top_p", tp, m)
    return pd.DataFrame(results)


def sweep_top_k(llm, prompts, eval_subset):
    results = []
    for tk in [10, 20, 40]:
        sp = SamplingParams(
            max_tokens=MAX_TOKENS, n=1,
            temperature=TEMPERATURE, top_p=0.95, top_k=tk, min_p=0.0,
            seed=SEED, stop=["<|im_end|>", "<|endoftext|>"],
        )
        m = run_generation(llm, prompts, sp, eval_subset)
        m["top_k"] = tk
        results.append(m)
        print_row("top_k", tk, m)
    return pd.DataFrame(results)


def sweep_min_p(llm, prompts, eval_subset):
    """Disable top_p/top_k so min_p is the active filter."""
    results = []
    for mp in [0.0, 0.05, 0.1]:
        sp = SamplingParams(
            max_tokens=MAX_TOKENS, n=1,
            temperature=TEMPERATURE, top_p=1.0, top_k=-1, min_p=mp,
            seed=SEED, stop=["<|im_end|>", "<|endoftext|>"],
        )
        m = run_generation(llm, prompts, sp, eval_subset)
        m["min_p"] = mp
        results.append(m)
        print_row("min_p", mp, m)
    return pd.DataFrame(results)


def sweep_repetition_penalty(llm, prompts, eval_subset):
    results = []
    for rp in [1.0, 1.05, 1.1]:
        sp = SamplingParams(
            max_tokens=MAX_TOKENS, n=1,
            temperature=TEMPERATURE, top_p=0.95, top_k=20, min_p=0.0,
            repetition_penalty=rp,
            seed=SEED, stop=["<|im_end|>", "<|endoftext|>"],
        )
        m = run_generation(llm, prompts, sp, eval_subset)
        m["repetition_penalty"] = rp
        results.append(m)
        print_row("rep_penalty", rp, m)
    return pd.DataFrame(results)


def run_sampling_sweeps(eval_subset, output_path="sweep_results_sampling.xlsx"):
    """GROUP A: All sampling-param sweeps with a single LLM instance."""
    prompts = build_prompts(eval_subset)
    sampling_results = {}

    print("=" * 60)
    print("GROUP A: Loading LLM for sampling-param sweeps")
    print("=" * 60)
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        trust_remote_code=True,
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=0.92,
        max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
        max_num_seqs=MAX_NUM_SEQS,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
        disable_log_stats=False,
    )

    try:
        print("\n>>> SWEEP: max_tokens")
        sampling_results["max_tokens"] = sweep_max_tokens(llm, prompts, eval_subset)

        print("\n>>> SWEEP: temperature")
        sampling_results["temperature"] = sweep_temperature(llm, prompts, eval_subset)

        print("\n>>> SWEEP: top_p")
        sampling_results["top_p"] = sweep_top_p(llm, prompts, eval_subset)

        print("\n>>> SWEEP: top_k")
        sampling_results["top_k"] = sweep_top_k(llm, prompts, eval_subset)

        print("\n>>> SWEEP: min_p")
        sampling_results["min_p"] = sweep_min_p(llm, prompts, eval_subset)

        print("\n>>> SWEEP: repetition_penalty")
        sampling_results["repetition_penalty"] = sweep_repetition_penalty(llm, prompts, eval_subset)

    finally:
        cleanup_llm(llm)

    with pd.ExcelWriter(output_path) as writer:
        for name, df in sampling_results.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)
    print(f"\nGroup A saved: {output_path}")

    print("\n=== GROUP A SUMMARY ===")
    for name, df in sampling_results.items():
        print(f"\n--- {name} ---")
        print(df.to_markdown(index=False))

    return sampling_results


# ============================================================
# GROUP B: SERVING-PARAM SWEEPS (LLM reloads per config)
# ============================================================
# Each sweep isolates ONE serving param while holding the other two
# at the baseline.
# ============================================================

# Sampling params held constant across all serving sweeps
SERVING_SAMPLING_PARAMS = SamplingParams(
    max_tokens=MAX_TOKENS, n=1,
    temperature=TEMPERATURE, top_p=0.95, top_k=20, min_p=0.0,
    seed=SEED,
    stop=["<|im_end|>", "<|endoftext|>"],
)


def _run_one_serving_config(prompts, eval_subset, n_seqs, n_batched, model_len,
                            sampling_params):
    """
    Load LLM with the given serving config, run eval, capture metrics,
    release LLM. Returns one result dict.
    """
    print(f"\n--- seqs={n_seqs}, batched={n_batched}, model_len={model_len} ---")
    llm = None
    try:
        llm = LLM(
            model=MODEL_ID,
            dtype="bfloat16",
            trust_remote_code=True,
            max_model_len=model_len,
            gpu_memory_utilization=0.92,
            max_num_batched_tokens=n_batched,
            max_num_seqs=n_seqs,
            enable_prefix_caching=True,
            enable_chunked_prefill=True,
            disable_log_stats=False,
        )

        t0 = time.time()
        outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=True)
        elapsed = time.time() - t0

        total_tokens = sum(len(s.token_ids) for out in outputs for s in out.outputs)
        throughput = total_tokens / elapsed

        responses = [out.outputs[0].text.strip() for out in outputs]
        acc, _ = evaluate(eval_subset, responses)

        result = {
            "max_num_seqs": n_seqs,
            "max_num_batched_tokens": n_batched,
            "max_model_len": model_len,
            "wall_time_s": elapsed,
            "throughput_tok_s": throughput,
            "accuracy": acc,
            "status": "OK",
        }
        print(f"  → time={elapsed:.1f}s, throughput={throughput:.0f} tok/s, acc={acc:.3f}")
        return result

    except Exception as e:
        print(f"  → FAILED: {e}")
        return {
            "max_num_seqs": n_seqs,
            "max_num_batched_tokens": n_batched,
            "max_model_len": model_len,
            "status": f"FAIL: {type(e).__name__}: {str(e)[:80]}",
        }

    finally:
        if llm is not None:
            cleanup_llm(llm)


def sweep_max_num_seqs(prompts, eval_subset):
    """
    Vary max_num_seqs; hold max_num_batched_tokens=16384, max_model_len=16384.
    Tests: concurrency-driven throughput vs. KV cache pressure.
      low (8):  underutilized, no preemption
      med (32): balanced baseline
      high (64): risks KV preemption with long thinking traces
    """
    print("\n>>> SWEEP B1: max_num_seqs (isolated)")
    results = []
    for n_seqs in [8, 32, 64]:
        r = _run_one_serving_config(
            prompts, eval_subset,
            n_seqs=n_seqs,
            n_batched=MAX_NUM_BATCHED_TOKENS,
            model_len=MAX_MODEL_LEN,
            sampling_params=SERVING_SAMPLING_PARAMS,
        )
        r["sweep_var"] = "max_num_seqs"
        results.append(r)
    return pd.DataFrame(results)


def sweep_max_num_batched_tokens(prompts, eval_subset):
    """
    Vary max_num_batched_tokens; hold max_num_seqs=32, max_model_len=16384.
    Tests: scheduler per-iteration budget — affects prefill chunking.
      low (4096):  forces aggressive chunking of long prefills
      med (8192):  moderate chunking
      high (16384): no chunking for max-length prefills
    Note: when n_batched < model_len, prefills get auto-chunked across
    iterations, which interleaves with concurrent decodes.
    """
    print("\n>>> SWEEP B2: max_num_batched_tokens (isolated)")
    results = []
    for n_batched in [4096, 8192, 16384]:
        r = _run_one_serving_config(
            prompts, eval_subset,
            n_seqs=MAX_NUM_SEQS,
            n_batched=n_batched,
            model_len=MAX_MODEL_LEN,
            sampling_params=SERVING_SAMPLING_PARAMS,
        )
        r["sweep_var"] = "max_num_batched_tokens"
        results.append(r)
    return pd.DataFrame(results)


def sweep_max_model_len(prompts, eval_subset):
    """
    Vary max_model_len; hold max_num_seqs=32, max_num_batched_tokens=16384.
    Tests: KV cache pre-allocation footprint vs. usable context.
      low (8192):  smaller KV blocks, more sequences fit; rejects long prompts
      med (12288): middle ground
      high (16384): full context, larger per-seq KV reservation
    Note: max_num_batched_tokens (16384) ≥ all model_len values, so
    prefill chunking behavior stays constant across this sweep.
    """
    print("\n>>> SWEEP B3: max_model_len (isolated)")
    results = []
    for model_len in [8192, 12288, 16384]:
        r = _run_one_serving_config(
            prompts, eval_subset,
            n_seqs=MAX_NUM_SEQS,
            n_batched=MAX_NUM_BATCHED_TOKENS,
            model_len=model_len,
            sampling_params=SERVING_SAMPLING_PARAMS,
        )
        r["sweep_var"] = "max_model_len"
        results.append(r)
    return pd.DataFrame(results)


def run_serving_sweeps(eval_subset, output_path="sweep_results_serving.xlsx"):
    """
    GROUP B: Three isolated serving-param sweeps.
    Each sweep does its own LLM reloads; failures in one don't block others.
    """
    prompts = build_prompts(eval_subset)
    serving_results = {}

    print("=" * 60)
    print("GROUP B: Serving-param sweeps (LLM reloads per config)")
    print(f"Baseline held: seqs={MAX_NUM_SEQS}, "
          f"batched={MAX_NUM_BATCHED_TOKENS}, model_len={MAX_MODEL_LEN}")
    print("=" * 60)

    serving_results["max_num_seqs"] = sweep_max_num_seqs(prompts, eval_subset)
    serving_results["max_num_batched_tokens"] = sweep_max_num_batched_tokens(prompts, eval_subset)
    serving_results["max_model_len"] = sweep_max_model_len(prompts, eval_subset)

    with pd.ExcelWriter(output_path) as writer:
        for name, df in serving_results.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)
    print(f"\nGroup B saved: {output_path}")

    print("\n=== GROUP B SUMMARY ===")
    for name, df in serving_results.items():
        print(f"\n--- {name} ---")
        print(df.to_markdown(index=False))

    return serving_results

In [29]:
acc, flags = evaluate(EVAL_SUBSET, ["test"] * len(EVAL_SUBSET))
print(type(flags))
print(type(flags[0]) if flags else "empty")
print(flags[0] if flags else "")

<class 'list'>
<class 'dict'>
{'id': 107, 'correct': False}


In [28]:
sampling_results = run_sampling_sweeps(EVAL_SUBSET)

GROUP A: Loading LLM for sampling-param sweeps
INFO 05-09 16:59:16 config.py:549] This model supports multiple tasks: {'reward', 'generate', 'score', 'classify', 'embed'}. Defaulting to 'generate'.
INFO 05-09 16:59:16 config.py:1555] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 05-09 16:59:16 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_tr

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-09 16:59:23 model_runner.py:1115] Loading model weights took 7.5454 GB
INFO 05-09 16:59:24 worker.py:267] Memory profiling takes 1.25 seconds
INFO 05-09 16:59:24 worker.py:267] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.92) = 72.91GiB
INFO 05-09 16:59:24 worker.py:267] model weights take 7.55GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 1.21GiB; the rest of the memory reserved for KV Cache is 64.15GiB.
INFO 05-09 16:59:25 executor_base.py:111] # cuda blocks: 29197, # CPU blocks: 1820
INFO 05-09 16:59:25 executor_base.py:116] Maximum concurrency for 16384 tokens per request: 28.51x
INFO 05-09 16:59:25 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 7/7 [00:07<00:00,  1.10s/it]

INFO 05-09 16:59:33 model_runner.py:1562] Graph capturing finished in 8 secs, took 0.03 GiB
INFO 05-09 16:59:33 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 10.08 seconds


...

INFO 05-09 17:01:54 metrics.py:455] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 168.5 tokens/s, Running: 2 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 1.7%, CPU KV cache usage: 0.0%.
INFO 05-09 17:01:54 metrics.py:471] Prefix cache hit rate: GPU: 15.84%, CPU: 0.00%
INFO 05-09 17:01:59 metrics.py:455] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 144.7 tokens/s, Running: 2 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 1.8%, CPU KV cache usage: 0.0%.
INFO 05-09 17:01:59 metrics.py:471] Prefix cache hit rate: GPU: 15.84%, CPU: 0.00%
INFO 05-09 17:02:04 metrics.py:455] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 143.7 tokens/s, Running: 2 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 2.0%, CPU KV cache usage: 0.0%.
INFO 05-09 17:02:04 metrics.py:471] Prefix cache hit rate: GPU: 15.84%, CPU: 0.00%
INFO 05-09 17:02:09 metrics.py:455] Avg prompt throughput: 0.0 tokens/s, Avg generation throug

Processed prompts:  98%|█████████▊| 49/50 [02:38<00:07,  7.60s/it, est. speed input: 83.31 toks/s, output: 940.32 toks/s] 

INFO 05-09 17:02:14 metrics.py:455] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 116.2 tokens/s, Running: 1 reqs, Swapped: 0 reqs, Pending: 0 reqs, GPU KV cache usage: 1.2%, CPU KV cache usage: 0.0%.
INFO 05-09 17:02:14 metrics.py:471] Prefix cache hit rate: GPU: 15.84%, CPU: 0.00%


Processed prompts: 100%|██████████| 50/50 [02:40<00:00,  3.20s/it, est. speed input: 92.98 toks/s, output: 953.75 toks/s]


TypeError: float() argument must be a string or a real number, not 'dict'

In [ ]:
serving_results = run_serving_sweeps(EVAL_SUBSET)

In [ ]:
# N_SAMPLES = 4

# sampling_sc = SamplingParams(
#     n=N_SAMPLES,                # vLLM samples N times in one call
#     max_tokens=MAX_TOKENS,
#     temperature=TEMPERATURE,            # bumped from 0.6 for more diversity
#     top_p=0.95,
#     top_k=20,
# )

# prompts = []
# for item in EVAL_SUBSET:
#     sys_p, _ = build_prompt(item["question"], item.get("options"))
#     prompts.append(make_prompt_with_system(item, sys_p))

# print(f"Generating {N_SAMPLES} samples × {len(prompts)} prompts...")
# outputs = llm.generate(prompts, sampling_params=sampling_sc, use_tqdm=True)

# # Each output has N completions in .outputs
# responses_grouped = [[o.text.strip() for o in out.outputs] for out in outputs]

# acc_sc, details_sc = evaluate(EVAL_SUBSET, responses_grouped)
# print(f"\nSelf-consistency (n={N_SAMPLES}): {acc_sc*100:.1f}%")

# # Save for analysis
# with open(EXPERIMENT_DIR / f"self_consistency_n{N_SAMPLES}.jsonl", "w") as f:
#     for d in details_sc:
#         f.write(json.dumps(d) + "\n")

In [ ]:
# FEWSHOT_FREEFORM = """Example 1:
# Question: What is the sum of the first 5 positive integers?
# Solution: We compute 1 + 2 + 3 + 4 + 5 = 15. The answer is \\boxed{15}.

# Example 2:
# Question: If x + 3 = 10, what is 2x?
# Solution: From x + 3 = 10, we get x = 7. So 2x = 14. The answer is \\boxed{14}.

# Now solve this problem:
# """

# FEWSHOT_MCQ = """Example 1:
# Question: What is 2 + 2?

# Options:
# A. 3
# B. 4
# C. 5
# D. 6

# Answer: \\boxed{B}

# Now answer this question:
# """

# prompts = []
# for item in EVAL_SUBSET:
#     sys_p, _ = build_prompt(item["question"], item.get("options"))
#     prefix = FEWSHOT_MCQ if item.get("options") else FEWSHOT_FREEFORM
#     prompts.append(make_prompt_with_system(item, sys_p, user_prefix=prefix))

# sampling_fs = SamplingParams(
#     max_tokens=12288, temperature=0.6, top_p=0.95, top_k=20,
# )
# outputs = llm.generate(prompts, sampling_params=sampling_fs, use_tqdm=True)
# responses = [out.outputs[0].text.strip() for out in outputs]

# acc_fs, details_fs = evaluate(EVAL_SUBSET, responses)
# print(f"\nFew-shot: {acc_fs*100:.1f}%")

# with open(EXPERIMENT_DIR / "fewshot.jsonl", "w") as f:
#     for d in details_fs:
#         f.write(json.dumps(d) + "\n")

In [ ]:
# REFLECT_SYSTEM = (
#     "You are an expert mathematician verifying a previous solution. "
#     "Read the problem and the candidate solution. If the solution is correct, restate the answer in \\boxed{}. "
#     "If you find an error, solve it correctly and put your final answer in \\boxed{}."
# )

# # Step 1: get initial answers
# sampling_init = SamplingParams(max_tokens=12288, temperature=0.6, top_p=0.95, top_k=20)
# init_prompts = [make_prompt_with_system(item, build_prompt(item["question"], item.get("options"))[0])
#                 for item in EVAL_SUBSET]
# init_outputs = llm.generate(init_prompts, sampling_params=sampling_init, use_tqdm=True)
# init_responses = [out.outputs[0].text.strip() for out in init_outputs]

# # Step 2: ask for verification
# verify_prompts = []
# for item, init_resp in zip(EVAL_SUBSET, init_responses):
#     _, original_user = build_prompt(item["question"], item.get("options"))
#     verify_user = (f"Problem:\n{original_user}\n\n"
#                    f"Candidate solution:\n{init_resp[-2000:]}\n\n"  # last 2k chars to avoid context blow-up
#                    f"Verify the answer. If wrong, solve correctly. Final answer in \\boxed{{}}.")
#     verify_prompts.append(tokenizer.apply_chat_template(
#         [{"role": "system", "content": REFLECT_SYSTEM},
#          {"role": "user", "content": verify_user}],
#         tokenize=False, add_generation_prompt=True,
#     ))

# verify_outputs = llm.generate(verify_prompts, sampling_params=sampling_init, use_tqdm=True)
# verify_responses = [out.outputs[0].text.strip() for out in verify_outputs]

# acc_reflect, details_reflect = evaluate(EVAL_SUBSET, verify_responses)
# print(f"\nReflection: {acc_reflect*100:.1f}%")

# # Compare against baseline (no reflection) on same items
# acc_base, _ = evaluate(EVAL_SUBSET, init_responses)
# print(f"  Baseline on same items: {acc_base*100:.1f}%")
# print(f"  Delta: {(acc_reflect - acc_base)*100:+.1f}%")

# with open(EXPERIMENT_DIR / "reflection.jsonl", "w") as f:
#     for d in details_reflect:
#         f.write(json.dumps(d) + "\n")

In [ ]:
# CoT_SYSTEM = (
#     "You are an expert mathematician. Think step by step, showing all reasoning. "
#     "After each major step, briefly verify it before continuing. "
#     "Put your final answer in \\boxed{}."
# )

# prompts = []
# for item in EVAL_SUBSET:
#     sys_p_to_use = CoT_SYSTEM
#     if item.get("options"):
#         # For MCQ, append the MCQ instruction
#         sys_p_to_use = CoT_SYSTEM + " For multiple choice, output only the letter inside \\boxed{}."
#     prompts.append(make_prompt_with_system(item, sys_p_to_use))

# outputs = llm.generate(prompts, sampling_params=sampling_init, use_tqdm=True)
# responses = [out.outputs[0].text.strip() for out in outputs]

# acc_cot, _ = evaluate(EVAL_SUBSET, responses)
# print(f"\nCoT prompt variant: {acc_cot*100:.1f}%")

In [ ]:
# print("=" * 50)
# print("EXPERIMENT SUMMARY")
# print("=" * 50)

# # Run baseline if you don't have it
# print(f"  Baseline (single, T=0.6):   {acc_base*100:5.1f}%")
# print(f"  Self-consistency (n=4):     {acc_sc*100:5.1f}%")
# # print(f"  Few-shot:                   {acc_fs*100:5.1f}%")
# print(f"  Reflection:                 {acc_reflect*100:5.1f}%")
# # print(f"  CoT prompt variant:         {acc_cot*100:5.1f}%")

In [ ]:
id_list = []
for item in EVAL_SUBSET:
  id_list.append(item['id'])

print(id_list)